In [1]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Convert data from byte into datatpyes
def convert_from_byte(byte_dict):
    return {key.decode('utf-8'): value.decode('utf-8') for key, value in byte_dict.items()}

In [3]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [4]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [ ]:
# Import datasets
# datasets = {}
# dataset_names = ['clfever', 'phemeplus', 'vitc']
# for dataset_name in dataset_names:
#     with open(f'{dataset_name}.json') as f:
#         datasets[dataset_name] = json.load(f)

In [ ]:
# Only do this once
# Import VITC daraset
with open("vitc_evaluation_sup_ref.json") as f:
    vitc = json.load(f)

# Randomise order
np.random.shuffle(vitc)
# Test labels
labels = []
for claim in vitc:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([217, 283]))

In [ ]:
# Only do this once
# Populate Vercel KV with vitc datasets
for datapoint in vitc:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': datapoint['label']
    })

In [11]:
vitc_ids = [datapoint['claim_id'] for datapoint in vitc]

In [12]:
# Save ids as json 
with open('vitc_ids.json', 'w') as f:
    json.dump(vitc_ids, f)

In [13]:
# OK after randomising once at so on, use vitc ids saved in json
with open('vitc_ids.json') as f:
    vitc_ids = json.load(f)

In [15]:
# Create three batches for VITC
# 100 samples are included in all batches to test for inter-rater reliability

repeated_samples = vitc_ids[:100]

vitc_batches = {
    'vitc_repeated': repeated_samples 
}
start_index = 100
for i in range(3):
    end_index = start_index + ((len(vitc_ids) - 100) // 3) if i < 2 else len(vitc_ids)
    print(f'{start_index} - {end_index}')
    unique_samples = vitc_ids[start_index:end_index]
    start_index = end_index
    vitc_batches[f'vitc{i+1}_workpackage1'] = unique_samples[:15]
    vitc_batches[f'vitc{i+1}_workpackage3'] = unique_samples[15:]

for key in vitc_batches:
    print(key, len(vitc_batches[key])) 

100 - 233
233 - 366
366 - 500
vitc_repeated 100
vitc1_workpackage1 15
vitc1_workpackage3 118
vitc2_workpackage1 15
vitc2_workpackage3 118
vitc3_workpackage1 15
vitc3_workpackage3 119


In [16]:
# Populate vercel KV with VITC batches
for batch_id in vitc_batches.keys():
    claim_ids = vitc_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [18]:
# Assign batches to annotators
vitc_annotators = {
    'test': 'vitc1',
    'mahmud': 'vitc2',
    'sara': 'vitc1',
}

In [19]:
# Upload annotators to Vercel KV
for annotator_id in vitc_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{vitc_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'vitc_repeated',
            'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [5]:
r.hset('mahmud', mapping={'stage': 'workpackage1', 'workpackage1_progress': 0, 'workpackage2_progress': 0, 'workpackage3_progress': 0})

0

In [20]:
test_data = r.hgetall('mahmud')
test_data = convert_from_byte(test_data)
test_data

{'vitc_13414': '["deductive","asd"]',
 'vitc_15978': '["deductive","asd"]',
 'workpackage1_progress': '0',
 'vitc_1038': '["abductive","asd"]',
 'vitc_4392': '["deductive","asd"]',
 'vitc_9219': '["deductive","asd"]',
 'vitc_13841': '["deductive","asd"]',
 'vitc_15855': '["deductive","Initially was an agricultural society "]',
 'vitc_8360': '["deductive","asd"]',
 'stage_to_batch': '{"workpackage1": "vitc2_workpackage1", "workpackage2": "vitc_repeated", "workpackage3": "vitc2_workpackage3"}',
 'vitc_925': '["deductive","asd"]',
 'vitc_11539': '["deductive","asd"]',
 'vitc_4172': '["deductive","asd"]',
 'workpackage3_progress': '0',
 'vitc_15319': '["deductive","asd"]',
 'stage': 'workpackage1',
 'vitc_2996': '["deductive","yeah sure, whatever"]',
 'workpackage2_progress': '0',
 'vitc_1017': '["deductive","asd"]',
 'vitc_15718': '["deductive","Valencia is the capital of the autonomous community of Valencia and the third largest city in Spain "]',
 'vitc_16163': '["deductive","Poetry is 

In [26]:
r.hgetall('vitc_15319')

{b'claim': b'Fibromyalgia can make it hard to get out of bed.',
 b'evidence': b'Fibromyalgia is a medical condition defined by the presence of chronic widespread pain, fatigue, waking unrefreshed, cognitive symptoms, lower abdominal pain or cramps, and depression. Other symptoms include insomnia and a general hypersensitivity. The cause of fibromyalgia is unknown, but is believed to involve a combination of genetic and environmental factors.',
 b'label': b'SUPPORTS'}

In [24]:
# Replace vitc_15319 with new datapoint, but keep id the same
replacement_data = {
    "claim": "Fibromyalgia can make it hard to get out of bed.",
    "evidence": "Fibromyalgia is a medical condition defined by the presence of chronic widespread pain, fatigue, waking unrefreshed, cognitive symptoms, lower abdominal pain or cramps, and depression. Other symptoms include insomnia and a general hypersensitivity. The cause of fibromyalgia is unknown, but is believed to involve a combination of genetic and environmental factors.",
    "label": "SUPPORTS"
}

r.hset('vitc_15319', mapping=replacement_data)

0

In [274]:
# get ids of participants who completed the task
completed_batches = []
completed_participants = []
submissions = r.lrange('participants',0,-26)
for submission in submissions:
    # convert to dictionary from bytes
    submission = submission.decode('utf-8')
    submission = json.loads(submission)

    if submission["stage"] == "annotation":
        completed_batches.append(submission["batchId"])
        completed_participants.append(submission["participant"])

In [280]:
len(completed_batches)

15

In [275]:
np.unique(completed_batches, return_counts=True)

(array(['batch_vitc_10', 'batch_vitc_11', 'batch_vitc_12', 'batch_vitc_14',
        'batch_vitc_16', 'batch_vitc_17', 'batch_vitc_18', 'batch_vitc_19',
        'batch_vitc_2', 'batch_vitc_3', 'batch_vitc_4', 'batch_vitc_5',
        'batch_vitc_6', 'batch_vitc_8', 'batch_vitc_9'], dtype='<U13'),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [287]:
vitc_queue = [id for id in vitc_batches.keys() if id not in completed_batches]
vitc_queue

['batch_vitc_1',
 'batch_vitc_7',
 'batch_vitc_13',
 'batch_vitc_15',
 'batch_vitc_20']

In [289]:
# delete queue and then only add remaining batches
r.delete('queue')
r.lpush('queue', *vitc_queue)

5

In [9]:
r.rpush('queue', *["batch_vitc_1", "batch_vitc_2", "batch_vitc_3", "batch_vitc_4"])

20

In [11]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

18
[b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4']


In [277]:
dataset = []
for participant in completed_participants:
    submission = r.hgetall(participant)
    submission = convert_from_byte(submission)
    for key in submission:
        if "asses" not in key:
            data = vitc_dict[key]
            data['reasoning'] = submission[key]
            data['participant'] = participant
            dataset.append(data)


In [278]:
dataset = pd.DataFrame(dataset)
dataset.to_json('dataset.json', orient='records', lines=True)

In [279]:
dataset['reasoning'].value_counts()

reasoning
deductive    231
abductive    144
Name: count, dtype: int64

## Phemplus

In [150]:
phemeplus_annotators = {
    'bleiz': 'phemeplus1',
    'nelly': 'phemeplus2',
    'yazhou': 'phemeplus3'
}

In [135]:


# Import phemeplus dataset
with open("phemeplus_incomplete_15-11-24.json") as f:
    phemeplus = json.load(f)

# Randomise order
np.random.shuffle(phemeplus)
# Test labels
labels = []
for claim in phemeplus:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array([False,  True]), array([ 91, 202]))

In [124]:
# Populate Vercel KV with phemeplus datasets
for datapoint in phemeplus:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': "true" if datapoint['label'] else "false"
    })

In [138]:
phemeplus_ids = [datapoint['claim_id'] for datapoint in phemeplus]

In [145]:
# Create three batches for PHEMEPLUS workpackage 1 (phemeplus is incomplete so far)
repeated_samples = phemeplus_ids[:100]
workpackage1_samples1 = phemeplus_ids[100:115]
workpackage1_samples2 = phemeplus_ids[115:130]
workpackage1_samples3 = phemeplus_ids[130:145]

phemeplus_batches = {
    'phemeplus_repeated': repeated_samples,
    'phemeplus1_workpackage1': workpackage1_samples1,
    'phemeplus2_workpackage1': workpackage1_samples2,
    'phemeplus3_workpackage1': workpackage1_samples3
}

for key in phemeplus_batches:
    print(key, len(phemeplus_batches[key]))


phemeplus_repeated 100
phemeplus1_workpackage1 15
phemeplus2_workpackage1 15
phemeplus3_workpackage1 15


In [147]:
# Populate vercel KV with phemeplus batches
for batch_id in phemeplus_batches.keys():
    claim_ids = phemeplus_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [151]:
# Upload annotators to Vercel KV
for annotator_id in phemeplus_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{phemeplus_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'phemeplus_repeated',
            # 'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [9]:
r.hgetall('yazhou')

{b'pplus_72': b'["deductive","The deceased gunman has been identified as Michael Zehaf-Bibeau by CBS, ABC and CTV and CBC"]',
 b'pplus_233': b'["deductive"," they believed the 28-year-old co-pilot of the Germanwings jet, Andreas Lubitz, had deliberately slammed a jet into the French Alps"]',
 b'pplus_247': b'["deductive","Uber Sydney changed tack and is currently offering free rides for passengers trying to leave the city"]',
 b'workpackage3_progress': b'0',
 b'pplus_13': b'["deductive","direct summary"]',
 b'pplus_27': b'["deductive","Parliament\xe2\x80\x99s sergeant-at-arms Kevin Vickers, being labelled a hero for shooting a suspected terrorist in the House of Commons"]',
 b'pplus_141': b'["deductive","direct mapping"]',
 b'pplus_231': b'["deductive","should be false evidence 27 years old"]',
 b'pplus_162': b'["deductive","12 -2 = 10"]',
 b'pplus_149': b'["deductive","The hostages who died were a 34-year-old man and a 38-year-old woman"]',
 b'workpackage2_progress': b'39',
 b'pplus_1

## Climate Fever

In [6]:
# Import VITC daraset
with open("clfever.json") as f:
    clfever = json.load(f)

# Randomise order
np.random.shuffle(clfever)
# Test labels
labels = []
for claim in clfever:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([6, 6]))